In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from capstone.config import FEATURES, NUMERIC, PLOT, STYLES
from capstone.data import load_recipes
from capstone.models import affinity, build_graph, centroids, distances, fit_centroids, predict, styles
from capstone.utilities import setup_logging

setup_logging("WARNING")

pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
COLORS = PLOT.style_colors

In [ ]:
df = load_recipes()
X, y = df[FEATURES], df["style"]
model = fit_centroids(X, y)
centroids(model)

In [ ]:
# Centroids in original units: numeric = class mean
df.groupby("style")[NUMERIC].mean().loc[STYLES]

In [ ]:
cat_profile = pd.concat(
    [pd.crosstab(df["style"], df[col], normalize="index").add_prefix(f"{col}=") for col in ["flour_type", "yeast_type", "fat_type", "sugar_type"]],
    axis=1,
).loc[STYLES]
cat_profile

In [ ]:
dist = distances(model, X)
aff = affinity(model, X)
result = pd.concat([y.rename("true"), predict(model, X).rename("predicted"), dist.add_prefix("dist_"), aff.add_prefix("aff_")], axis=1)
result.head(10)

In [ ]:
print("resubstitution accuracy:", (result["true"] == result["predicted"]).mean().round(3))
pd.crosstab(result["true"], result["predicted"])

In [ ]:
result.loc[result["true"] != result["predicted"]]

In [ ]:
cdist = pd.DataFrame(
    np.linalg.norm(centroids(model).values[:, None] - centroids(model).values[None], axis=-1),
    index=styles(model), columns=styles(model),
)
cdist

In [ ]:
G = build_graph(model, X, y, k=3)
print(G)
print(pd.Series(dict(G.nodes(data="kind"))).value_counts())
print(pd.Series([d["kind"] for *_, d in G.edges(data=True)]).value_counts())

In [ ]:
# Node/edge table views
nodes = pd.DataFrame.from_dict(dict(G.nodes(data=True)), orient="index")
edges = nx.to_pandas_edgelist(G)
display(nodes.head())
edges.sort_values("distance").head(10)

In [ ]:
# Recipe -> centroid network: each recipe linked to every style centroid, edge width = affinity
pos = nx.spring_layout(G, seed=0, weight=None, k=0.6)
ce = [(u, v) for u, v, d in G.edges(data=True) if d["kind"] == "centroid"]
fig, ax = plt.subplots(figsize=(9, 8))
nx.draw_networkx_edges(G, pos, edgelist=ce, width=[0.2 + 4 * G.edges[e]["affinity"] for e in ce], alpha=0.35, ax=ax)
recipes = [n for n, d in G.nodes(data=True) if d["kind"] == "recipe"]
nx.draw_networkx_nodes(G, pos, nodelist=recipes, node_color=[COLORS[G.nodes[n]["style"]] for n in recipes], node_size=60, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=STYLES, node_color=[COLORS[s] for s in STYLES], node_size=900, node_shape="s", edgecolors="k", ax=ax)
nx.draw_networkx_labels(G, pos, labels={s: s for s in STYLES}, font_size=8, ax=ax)
ax.axis("off")
fig.tight_layout()

In [ ]:
# Recipe -> recipe kNN network in the preprocessed feature space, centroids hidden
H = G.subgraph(recipes).copy()
H.remove_edges_from([(u, v) for u, v, d in list(H.edges(data=True)) if d["kind"] != "neighbor"])
pos_h = nx.spring_layout(H, seed=0, weight=None)
fig, ax = plt.subplots(figsize=(9, 8))
same = [(u, v) for u, v, d in H.edges(data=True) if d["same_style"]]
diff = [(u, v) for u, v, d in H.edges(data=True) if not d["same_style"]]
nx.draw_networkx_edges(H, pos_h, edgelist=same, alpha=0.4, ax=ax)
nx.draw_networkx_edges(H, pos_h, edgelist=diff, edge_color="k", style="dashed", alpha=0.8, ax=ax)
nx.draw_networkx_nodes(H, pos_h, node_color=[COLORS[H.nodes[n]["style"]] for n in H], node_size=80, ax=ax)
ax.axis("off")
fig.tight_layout()

In [ ]:
# Graph metrics: how class-pure is the neighbor network, and which recipes bridge styles
H_edges = pd.DataFrame([{"u": u, "v": v, **d} for u, v, d in H.edges(data=True)])
print("fraction of neighbor edges within the same style:", H_edges["same_style"].mean().round(3))
bridge = pd.Series({n: sum(not H.edges[n, m]["same_style"] for m in H[n]) / max(H.degree(n), 1) for n in H}, name="cross_style_edge_frac")
metrics = pd.DataFrame({
    "style": pd.Series(dict(H.nodes(data="style"))),
    "degree": pd.Series(dict(H.degree())),
    "cross_style_edge_frac": bridge,
    "betweenness": pd.Series(nx.betweenness_centrality(H)),
})
metrics.sort_values("cross_style_edge_frac", ascending=False).head(10)

In [ ]:
nx.community.modularity(H, [{n for n in H if H.nodes[n]["style"] == s} for s in STYLES])